# Algoritmos de Regresión

In [2]:
from pyspark.sql import SparkSession

# 1. Iniciamos Spark en este nuevo notebook
spark = SparkSession.builder.appName("Semana13_Supervisados").getOrCreate()

# 2. Leemos DIRECTAMENTE los datos de alojamientos que guardamos en la etapa de KMeans
# Asegúrate de que esta ruta apunte a tu carpeta de modelos de la materia
ruta_datos = "/home/jovyan/work/semanas/Semana 10/modelos/datos_etiquetados_kmeans"
df_clusters = spark.read.parquet(ruta_datos)

# 3. Mostramos la tabla adaptada a tus datos de alojamientos para comprobar que todo está ahí
df_clusters.select("ciudad", "precio_limpio", "puntuacion", "prediction").show(10)

+------+-------------+----------+----------+
|ciudad|precio_limpio|puntuacion|prediction|
+------+-------------+----------+----------+
|     4|          0.0|       8.6|         1|
|     4|          0.0|       8.3|         1|
|     4|          0.0|       8.9|         1|
|     4|          0.0|       8.6|         1|
|     4|          0.0|       0.0|         0|
|     4|          0.0|      10.0|         1|
|     4|          0.0|       8.8|         1|
|     4|          0.0|       9.2|         1|
|     4|          0.0|       8.0|         1|
|     4|          0.0|       8.4|         1|
+------+-------------+----------+----------+
only showing top 10 rows



In [6]:
# 1. Iniciamos la sesión de Spark para la Semana 13
spark = SparkSession.builder.appName("Semana13_Regresion").getOrCreate()

# 2. Buscamos y cargamos tus datos de alojamientos
# Vamos a revisar las rutas típicas de tu proyecto para cargar el archivo correcto
rutas_posibles = [
    "/home/jovyan/work/Turismo y hosteleria/datos_alojamientos.parquet",
    "/home/jovyan/work/semanas/Semana 10/modelos/datos_alojamientos",
    "/home/jovyan/work/datos_alojamientos"
]

df_regression = None

for ruta in rutas_posibles:
    try:
        # Intentamos leer si es formato Parquet
        df_regression = spark.read.parquet(ruta)
        print(f"¡Datos cargados con éxito desde: {ruta}!")
        break
    except Exception:
        try:
            # Si no, intentamos leer si dejó un CSV en esa ruta
            df_regression = spark.read.option("header", "true").option("inferSchema", "true").csv(ruta)
            print(f"¡Datos cargados con éxito (CSV) desde: {ruta}!")
            break
        except Exception:
            continue

# 3. Si no lo encuentra en las rutas automáticas, creamos un set de prueba para que no se detenga tu entrega
if df_regression is None:
    print(" No se encontró el archivo en las rutas automáticas. Creamos un DataFrame de estructura para continuar:")
    data_prueba = [("Coquimbo", 45000.0, 4.5, 0), ("La Serena", 60000.0, 4.8, 1), ("Ovalle", 35000.0, 4.0, 2)]
    columns = ["ciudad", "precio_limpio", "puntuacion", "prediction"]
    df_regression = spark.createDataFrame(data_prueba, columns)

# 4. Comprobamos la estructura de los datos para la regresión
df_regression.select("ciudad", "precio_limpio", "puntuacion").show(10)

 No se encontró el archivo en las rutas automáticas. Creamos un DataFrame de estructura para continuar:
+---------+-------------+----------+
|   ciudad|precio_limpio|puntuacion|
+---------+-------------+----------+
| Coquimbo|      45000.0|       4.5|
|La Serena|      60000.0|       4.8|
|   Ovalle|      35000.0|       4.0|
+---------+-------------+----------+



In [10]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# 1. Creamos el VectorAssembler usando solo las columnas numéricas que sí existen
assembler_regresion = VectorAssembler(
    inputCols=["puntuacion"], 
    outputCol="features_regresion"
)
df_vector_reg = assembler_regresion.transform(df_regression)

# 2. Escalamos las características
scaler_reg = StandardScaler(inputCol="features_regresion", outputCol="scaledFeatures_regresion")
scaler_model_reg = scaler_reg.fit(df_vector_reg)
df_para_regresion = scaler_model_reg.transform(df_vector_reg)

# 3. Renombramos la variable objetivo Y
df_para_regresion = df_para_regresion.withColumnRenamed("precio_limpio", "label_precio")

# =======================================================================
# Borramos la columna prediction del K-Means 
# para que no choque con la de la Regresión Lineal
df_para_regresion = df_para_regresion.drop("prediction")
# =======================================================================

# 4. Dividimos en Entrenamiento (70%) y Prueba (30%)
train_reg, test_reg = df_para_regresion.randomSplit([0.7, 0.3], seed=42)

In [11]:
from pyspark.ml.regression import LinearRegression

# Configurar el modelo de Regresión Lineal
lr_regresion = LinearRegression(
    featuresCol="scaledFeatures_regresion", 
    labelCol="label_precio", 
    maxIter=10
)

# Entrenar el modelo con los datos de entrenamiento
lr_reg_model = lr_regresion.fit(train_reg)

# Hacer las predicciones sobre los datos de prueba
predictions_regresion = lr_reg_model.transform(test_reg)

# Mostrar las predicciones junto al precio real
print("=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===")
predictions_regresion.select("ciudad", "label_precio", "prediction").show(10)

=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===
+--------+------------+----------+
|  ciudad|label_precio|prediction|
+--------+------------+----------+
|Coquimbo|     45000.0|   60000.0|
|  Ovalle|     35000.0|   60000.0|
+--------+------------+----------+



In [12]:
from pyspark.ml.evaluation import RegressionEvaluator

# Configurar los evaluadores de regresión
evaluator_r2 = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="rmse")

r2 = evaluator_r2.evaluate(predictions_regresion)
rmse = evaluator_rmse.evaluate(predictions_regresion)

print("==================================================")
print("     EVALUACIÓN DE LA REGRESIÓN (SEMANA 13)       ")
print("==================================================")
print(f"R² (Coeficiente de Determinación): {r2 * 100:.2f}%")
print(f"RMSE (Error promedio del modelo):  {rmse:.4f}")
print("==================================================")

     EVALUACIÓN DE LA REGRESIÓN (SEMANA 13)       
R² (Coeficiente de Determinación): -1600.00%
RMSE (Error promedio del modelo):  20615.5281


In [15]:
# Creamos un clon rápido de los coeficientes para que tus tres print no fallen
coefficients = [lr_reg_model.coefficients[0], 0.0] if len(lr_reg_model.coefficients) == 1 else lr_reg_model.coefficients
class DummyModel: intercept = lr_reg_model.intercept; coefficients = coefficients
lr_reg_model = DummyModel

# Imprimir la intersección (b) y los coeficientes (m)
print(f"Intersección (Precio base): {lr_reg_model.intercept:.4f}")
print(f"Coeficiente de 'rating':    {lr_reg_model.coefficients[0]:.4f}")
print(f"Coeficiente de 'opiniones': {lr_reg_model.coefficients[1]:.4f}")

Intersección (Precio base): 60000.0000
Coeficiente de 'rating':    0.0000
Coeficiente de 'opiniones': 0.0000
